# Simulador 14.2 — Rentabilidad y maximización

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arbouria/Libro-ACA-2026/blob/main/Simuladores/sim14_2_rent_max.ipynb)

Este simulador ilustra cómo cambia la rentabilidad total cuando un organismo distribuye su tiempo entre dos alternativas.

El objetivo pedagógico es mostrar la diferencia entre:

- **elección local**: mover tiempo hacia la alternativa con mayor rentabilidad marginal;
- **maximización global**: encontrar la distribución de tiempo que maximiza la suma de reforzadores obtenidos.

La versión está limpia: no incluye `metadata.widgets`, para evitar errores de renderizado en GitHub.


## Modelo

Sea \(p\) la proporción de tiempo dedicada a la alternativa A.  
La proporción dedicada a B es \(1-p\).

Usamos funciones de retroalimentación con rendimientos decrecientes:

\[
R_A(p) = A_{\max}(1-e^{-k_A p})
\]

\[
R_B(1-p) = B_{\max}(1-e^{-k_B(1-p)})
\]

La rentabilidad total es:

\[
R_T(p) = R_A(p) + R_B(1-p)
\]

La solución maximizadora se encuentra donde la rentabilidad total es máxima.  
Cuando el óptimo está en el interior, la condición equivale a igualar las rentabilidades marginales de A y B.


In [ ]:
# Importaciones y configuración para Colab/Jupyter

import numpy as np
import matplotlib.pyplot as plt

from IPython.display import display, Markdown
import ipywidgets as widgets

# En Google Colab, esto ayuda a que los widgets funcionen correctamente.
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False


In [ ]:
def reinforcers_A(p, Amax, kA):
    # Reforzadores obtenidos en A como función de la proporción de tiempo en A.
    p = np.asarray(p)
    return Amax * (1 - np.exp(-kA * p))


def reinforcers_B(p, Bmax, kB):
    # Reforzadores obtenidos en B como función de la proporción de tiempo en A.
    # Como p es tiempo en A, el tiempo en B es 1 - p.
    p = np.asarray(p)
    return Bmax * (1 - np.exp(-kB * (1 - p)))


def total_reinforcers(p, Amax, kA, Bmax, kB):
    # Reforzadores totales obtenidos con una asignación p a A.
    return reinforcers_A(p, Amax, kA) + reinforcers_B(p, Bmax, kB)


def marginal_A(p, Amax, kA):
    # Rentabilidad marginal de asignar un poco más de tiempo a A.
    p = np.asarray(p)
    return Amax * kA * np.exp(-kA * p)


def marginal_B(p, Bmax, kB):
    # Rentabilidad marginal de asignar un poco más de tiempo a B.
    p = np.asarray(p)
    return Bmax * kB * np.exp(-kB * (1 - p))


def find_optimum(Amax, kA, Bmax, kB, n_grid=2001):
    # Busca el máximo global en una malla fina.
    p_grid = np.linspace(0, 1, n_grid)
    rt = total_reinforcers(p_grid, Amax, kA, Bmax, kB)
    idx = np.argmax(rt)
    return p_grid[idx], rt[idx], p_grid, rt


In [ ]:
def simulate_adjustment(p0, Amax, kA, Bmax, kB, eta=0.08, n_steps=60):
    # Dinámica simple de mejoramiento:
    # si la rentabilidad marginal de A es mayor que la de B, aumenta p;
    # si la de B es mayor, disminuye p.
    p_values = [p0]
    for _ in range(n_steps):
        p = p_values[-1]
        difference = marginal_A(p, Amax, kA) - marginal_B(p, Bmax, kB)
        p_new = np.clip(p + eta * difference, 0, 1)
        p_values.append(float(p_new))
    return np.array(p_values)


def classify_direction(p_actual, p_opt):
    if abs(p_actual - p_opt) < 0.02:
        return "La asignación actual está muy cerca del máximo global."
    if p_actual < p_opt:
        return "Conviene aumentar la proporción de tiempo dedicada a A."
    return "Conviene disminuir la proporción de tiempo dedicada a A y aumentar B."


In [ ]:
def plot_simulator(Amax=40, kA=4.0, Bmax=35, kB=3.0, p_actual=0.50, eta=0.08, n_steps=60):
    p_opt, rt_opt, p_grid, rt = find_optimum(Amax, kA, Bmax, kB)

    RA = reinforcers_A(p_grid, Amax, kA)
    RB = reinforcers_B(p_grid, Bmax, kB)
    mA = marginal_A(p_grid, Amax, kA)
    mB = marginal_B(p_grid, Bmax, kB)

    rt_actual = total_reinforcers(p_actual, Amax, kA, Bmax, kB)
    mA_actual = marginal_A(p_actual, Amax, kA)
    mB_actual = marginal_B(p_actual, Bmax, kB)

    display(Markdown(f"""
### Resultado

- Proporción actual en A: **{p_actual:.2f}**
- Proporción maximizadora en A: **{p_opt:.2f}**
- Reforzadores totales con la asignación actual: **{rt_actual:.2f}**
- Reforzadores totales máximos: **{rt_opt:.2f}**
- Rentabilidad marginal de A en la asignación actual: **{mA_actual:.2f}**
- Rentabilidad marginal de B en la asignación actual: **{mB_actual:.2f}**

**Interpretación:** {classify_direction(p_actual, p_opt)}
"""))

    # Gráfica 1: reforzadores obtenidos
    plt.figure()
    plt.plot(p_grid, RA, label="Reforzadores en A")
    plt.plot(p_grid, RB, label="Reforzadores en B")
    plt.plot(p_grid, rt, label="Total")
    plt.axvline(p_actual, linestyle="--", label="Asignación actual")
    plt.axvline(p_opt, linestyle=":", label="Máximo global")
    plt.xlabel("Proporción de tiempo en A")
    plt.ylabel("Reforzadores obtenidos")
    plt.title("Rentabilidad total y asignación de tiempo")
    plt.legend()
    plt.show()

    # Gráfica 2: rentabilidades marginales
    plt.figure()
    plt.plot(p_grid, mA, label="Rentabilidad marginal de A")
    plt.plot(p_grid, mB, label="Rentabilidad marginal de B")
    plt.axvline(p_actual, linestyle="--", label="Asignación actual")
    plt.axvline(p_opt, linestyle=":", label="Máximo global")
    plt.xlabel("Proporción de tiempo en A")
    plt.ylabel("Rentabilidad marginal")
    plt.title("Condición local: comparar rentabilidades marginales")
    plt.legend()
    plt.show()

    # Gráfica 3: ajuste por mejoramiento
    trajectory = simulate_adjustment(p_actual, Amax, kA, Bmax, kB, eta=eta, n_steps=n_steps)
    plt.figure()
    plt.plot(np.arange(len(trajectory)), trajectory, marker="o")
    plt.axhline(p_opt, linestyle=":", label="Máximo global")
    plt.ylim(-0.02, 1.02)
    plt.xlabel("Iteración")
    plt.ylabel("Proporción de tiempo en A")
    plt.title("Ajuste local hacia la asignación maximizadora")
    plt.legend()
    plt.show()


## Ejecutar el simulador

Modifica los parámetros y observa cómo cambia la asignación maximizadora.

- \(A_{\max}\) y \(B_{\max}\): límite superior de reforzadores disponibles en cada alternativa.
- \(k_A\) y \(k_B\): rapidez con la que se obtienen reforzadores conforme se asigna tiempo a la alternativa.
- \(p_{actual}\): proporción actual de tiempo dedicada a A.
- \(\eta\): tamaño del ajuste local.


In [ ]:
widgets.interact(
    plot_simulator,
    Amax=widgets.FloatSlider(value=40, min=5, max=80, step=1, description="Amax"),
    kA=widgets.FloatSlider(value=4.0, min=0.2, max=10.0, step=0.1, description="kA"),
    Bmax=widgets.FloatSlider(value=35, min=5, max=80, step=1, description="Bmax"),
    kB=widgets.FloatSlider(value=3.0, min=0.2, max=10.0, step=0.1, description="kB"),
    p_actual=widgets.FloatSlider(value=0.50, min=0.0, max=1.0, step=0.01, description="p actual"),
    eta=widgets.FloatSlider(value=0.08, min=0.005, max=0.20, step=0.005, description="eta"),
    n_steps=widgets.IntSlider(value=60, min=5, max=150, step=5, description="pasos"),
);


## Actividades sugeridas

1. Cambia \(A_{\max}\) y \(B_{\max}\). ¿El máximo global siempre favorece a la alternativa con mayor valor máximo?
2. Cambia \(k_A\) y \(k_B\). ¿Qué ocurre cuando una alternativa entrega reforzadores más rápido al inicio, pero se satura antes?
3. Coloca \(p_{actual}\) lejos del óptimo. ¿La regla local de mejoramiento se mueve en la dirección correcta?
4. Busca un caso en el que el óptimo esté cerca de 0 o de 1. ¿Qué significa psicológicamente una solución de esquina?
